<a href="https://colab.research.google.com/github/S48avio/Tokenizers/blob/main/Byte_Level_BytePair_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Byte Based BPE

In [1]:
from collections import defaultdict


# Special marker: 256 = Ġ (space prefix), outside real byte range 0-255
SPACE_MARKER = 256


class ByteBPETokenizer:
    def __init__(self):
        self.merge_rules = []      # list of ((a,b), new_id) in order
        self.token_to_id = {}      # token_key → int id
        self.id_to_token = {}      # int id → token_key
        self.vocab = {}            # token_key → bytes (for decoding)

    def _get_corpus_vocab(self, corpus: str) -> dict:
        word_freq = defaultdict(int)
        words = corpus.split(' ')
        for i, word in enumerate(words):
            if not word:
                continue
            byte_seq = list(word.encode('utf-8'))
            if i > 0: # elimate adding special character to the first word on the corpus
                byte_seq = [SPACE_MARKER] + byte_seq
            word_freq[tuple(byte_seq)] += 1
        return dict(word_freq)

    def _get_pair_counts(self, vocab: dict) -> dict:
        counts = defaultdict(int)
        for word_tuple, freq in vocab.items():
            for i in range(len(word_tuple) - 1):
                counts[(word_tuple[i], word_tuple[i+1])] += freq
        return dict(counts)

    def _merge_vocab(self, pair: tuple, vocab: dict) -> dict:
        a, b = pair
        new_vocab = {}
        for word_tuple, freq in vocab.items():
            new_word = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple)-1 and word_tuple[i]==a and word_tuple[i+1]==b:
                    new_word.append(pair)   # merged token = the pair itself as key
                    i += 2
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            new_vocab[tuple(new_word)] = freq
        return new_vocab

    def _token_to_bytes(self, token_key) -> bytes:
        """Recursively resolve a token key to its raw bytes."""
        if isinstance(token_key, int):
            if token_key == SPACE_MARKER:
                return b' '          # Ġ = literal space
            return bytes([token_key])
        # It's a merged pair — resolve recursively
        a, b = token_key
        return self._token_to_bytes(a) + self._token_to_bytes(b)

    def train(self, corpus: str, num_merges: int):
        word_freq = self._get_corpus_vocab(corpus)

        # ── Base vocabulary: ALL 256 bytes + Ġ marker ──
        token_vocab = set()
        for i in range(256):
            token_vocab.add(i)
        token_vocab.add(SPACE_MARKER)

        # ── Merge loop ──
        for _ in range(num_merges):
            pair_counts = self._get_pair_counts(word_freq)
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            word_freq = self._merge_vocab(best_pair, word_freq)
            self.merge_rules.append(best_pair)
            token_vocab.add(best_pair)

        # ── Build lookup tables ──
        # Sort: ints first (base bytes), then tuples (merges) by depth
        base_tokens = sorted(t for t in token_vocab if isinstance(t, int))
        merge_tokens = [r for r in self.merge_rules]  # preserve merge order

        all_tokens = base_tokens + merge_tokens
        self.token_to_id = {t: i for i, t in enumerate(all_tokens)}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}

        # Pre-compute bytes for every token (for fast decoding)
        self.vocab = {t: self._token_to_bytes(t) for t in all_tokens}

    # ── Encoding ──────────────────────────────────────────────

    def _encode_word(self, byte_seq: list) -> list:
        """Apply merge rules in order to a byte sequence."""
        tokens = list(byte_seq)
        for pair in self.merge_rules:
            a, b = pair
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens)-1 and tokens[i]==a and tokens[i+1]==b:
                    new_tokens.append(pair)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens

    def encode(self, text: str) -> list[int]:
        """text → list of integer token IDs."""
        ids = []
        words = text.split(' ')
        for i, word in enumerate(words):
            if not word:
                continue
            byte_seq = list(word.encode('utf-8'))
            if i > 0:
                byte_seq = [SPACE_MARKER] + byte_seq
            tokens = self._encode_word(byte_seq)
            ids.extend(self.token_to_id[t] for t in tokens)
        return ids

    def tokenize(self, text: str) -> list[str]:
        """Return human-readable token strings (for debugging)."""
        result = []
        words = text.split(' ')
        for i, word in enumerate(words):
            if not word:
                continue
            byte_seq = list(word.encode('utf-8'))
            if i > 0:
                byte_seq = [SPACE_MARKER] + byte_seq
            tokens = self._encode_word(byte_seq)
            for t in tokens:
                b = self.vocab[t]
                result.append(b.decode('utf-8', errors='replace'))
        return result

    # ── Decoding ──────────────────────────────────────────────

    def decode(self, ids: list[int]) -> str:
        """list of integer IDs → original text string."""
        raw_bytes = b''.join(self.vocab[self.id_to_token[i]] for i in ids)
        return raw_bytes.decode('utf-8')


# ── Demo ──────────────────────────────────────────────────────
if __name__ == "__main__":
    corpus = "low lower newest widest lowest low low"
    tok = ByteBPETokenizer()
    tok.train(corpus, num_merges=20)

    tests = [
        "low lowest lower",
        "the cat sat on the mat",    # unknown in char BPE, fine here
        "café naïve",                # non-ASCII: works perfectly
        "hello 🔥",                  # emoji: 4 bytes, no problem
        "中文",                       # Chinese: multi-byte UTF-8
    ]

    print("=== Roundtrip tests ===\n")
    for text in tests:
        tokens  = tok.tokenize(text)
        ids     = tok.encode(text)
        decoded = tok.decode(ids)
        ok = "✓" if decoded == text else "✗"
        print(f"{ok}  '{text}'")
        print(f"   tokens : {tokens}")
        print(f"   decoded: '{decoded}'\n")

=== Roundtrip tests ===

✓  'low lowest lower'
   tokens : ['low', ' lowest', ' lower']
   decoded: 'low lowest lower'

✓  'the cat sat on the mat'
   tokens : ['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e', ' ', 'm', 'a', 't']
   decoded: 'the cat sat on the mat'

✓  'café naïve'
   tokens : ['c', 'a', 'f', '�', '�', ' n', 'a', '�', '�', 'v', 'e']
   decoded: 'café naïve'

✓  'hello 🔥'
   tokens : ['h', 'e', 'l', 'lo', ' ', '�', '�', '�', '�']
   decoded: 'hello 🔥'

✓  '中文'
   tokens : ['�', '�', '�', '�', '�', '�']
   decoded: '中文'

